# Day 15 · The Ultralytics Framework on T4

**Objective:** Install Ultralytics, load pretrained YOLOv8n, and run object-detection inference on an image from a fruit-and-veggie dataset.

## 1. Download a farm-related image

Download a fruit-and-veggie image dataset from Roboflow. We only need an image for inference today; no model training is performed.

In [ ]:
!pip -q install roboflow ultralytics

from getpass import getpass
from roboflow import Roboflow
import yaml
from pathlib import Path

ROBOFLOW_API_KEY = getpass("Enter your Roboflow API key: ")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("projects-gmksb").project("fruit-and-veggie")
dataset = project.version(4).download("yolov8")
DATA_DIR = Path(dataset.location)

with open(DATA_DIR / "data.yaml", "r") as file:
    data_config = yaml.safe_load(file)

class_names = data_config["names"]
if isinstance(class_names, dict):
    class_names = [class_names[i] for i in sorted(class_names)]

print("Dataset:", DATA_DIR)
print("Classes:", class_names)

## 2. Load YOLOv8n

YOLOv8n is the nano version of YOLOv8. It is a lightweight pretrained detector suitable for a quick inference demonstration.

In [ ]:
from ultralytics import YOLO
from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt

model = YOLO("yolov8n.pt")

## 3. Run inference on one dataset image

Choose one image from the dataset and let the pretrained model predict object classes and bounding boxes.

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png"}
image_paths = [p for p in DATA_DIR.rglob("*")
               if p.suffix.lower() in image_extensions and "images" in p.parts]

if not image_paths:
    raise FileNotFoundError("No images found in the downloaded dataset.")

image_path = image_paths[0]
results = model.predict(source=str(image_path), conf=0.25, imgsz=640, device=0 if __import__("torch").cuda.is_available() else "cpu")

annotated = results[0].plot()
plt.figure(figsize=(10, 7))
plt.imshow(annotated[:, :, ::-1])
plt.title("YOLOv8n Object Detection")
plt.axis("off")
plt.show()

print("Detected objects:", len(results[0].boxes))

## 4. Inspect predicted classes and confidence scores

In [ ]:
for box in results[0].boxes:
    class_id = int(box.cls.item())
    confidence = float(box.conf.item())
    print(f"{model.names[class_id]}: {confidence:.3f}")

### Day 15 · Key takeaway

Ultralytics provides a simple interface to load pretrained YOLO models and run inference. The model returns predicted classes, confidence scores, and bounding-box coordinates.